# Can Your Voice Predict Parkinson's Severity?
## OLS Regression on Vocal Biomarkers — UCI Telemonitoring Dataset

**Author:** Thibault Fieve  
**Dataset:** UCI Parkinson's Telemonitoring (ID 189)  


---

### Why This Project

I have two passions: data science and neurological diseases. This project sits exactly at the intersection of both.

Parkinson's disease affects over 10 million people worldwide. One of its earliest and most measurable symptoms is a change in the voice  patients gradually lose control of their pitch, amplitude, and vocal rhythm. This degradation can be captured by a simple microphone recording at home, without any clinic visit.

I wanted to test a simple question: **can we predict how severe a patient's motor symptoms are, just by analyzing their voice?** This is not just a modeling exercise, it has real implications for low-cost, non-invasive remote monitoring of a disease that is currently very expensive to track.


## 1. Imports

In [ ]:
from ucimlrepo import fetch_ucirepo 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

## 2. Dataset

The data comes from a real clinical study where **42 Parkinson's patients** wore recording devices at home over several months. Each recording session produced a set of acoustic measurements of the patient's voice.

The variable we want to predict is `motor_UPDRS` — a score assigned by a clinician that measures how severe the patient's motor symptoms are on a scale from 0 to 108. A higher score means worse symptoms.

| | |
|---|---|
| Total recordings | 5,875 |
| Patients | 42 |
| Vocal features | 22 acoustic measurements |
| Target variable | motor_UPDRS (motor symptom severity) |

**Source:** Tsanas, Little, McSharry & Ramig (2009),
*Accurate telemonitoring of Parkinson's disease progression
by non-invasive speech tests*,
IEEE Transactions on Biomedical Engineering.


In [ ]:
# Load the Parkinson's Telemonitoring dataset from the UCI repository
parkinsons_telemonitoring = fetch_ucirepo(id=189) 
x = parkinsons_telemonitoring.data.features
y = parkinsons_telemonitoring.data.targets 
print(parkinsons_telemonitoring.metadata)

## 3. Data Preparation

Before building the model, three steps are needed:

**Step 1 — Select the target variable**  
We use `motor_UPDRS` as the variable to predict.

**Step 2 — Standardize the features**  
Each feature is rescaled to have a mean of 0 and a standard deviation of 1. This allows us to compare the coefficients directly, a larger coefficient means a stronger impact, regardless of the original unit of measurement.

**Step 3 — Remove perfectly collinear variables**  
`Jitter:DDP` is mathematically equal to 3 × `Jitter:RAP`, and `Shimmer:DDA` is equal to 3 × `Shimmer:APQ3`. Keeping identical information twice would break the OLS matrix inversion, so we drop them.


In [ ]:
# Data preparation — standardize features and remove non-vocal variables
target_variable = y['motor_UPDRS']
features = x
features = (features - features.mean()) / features.std()
features['Intercept'] = 1
features = features.drop(columns=['age','test_time','Jitter:DDP','Shimmer:DDA'])
final_features        = np.array(features)
final_target_variable = np.array(target_variable)

print(f"Dataset shape : {final_features.shape[0]:,} observations x {final_features.shape[1]} features")
print(f"Target range  : [{final_target_variable.min():.1f}, {final_target_variable.max():.1f}]")

## 4. Removing Redundant Variables — VIF

Many vocal biomarkers in this dataset measure very similar things. For example, Shimmer, Shimmer:APQ3, Shimmer:APQ5 and Shimmer:APQ11 all capture amplitude instability in slightly different ways. Including all of them does not add useful information, it just creates noise in the coefficients.

The **Variance Inflation Factor (VIF)** detects this problem. It answers the question: *how much is this variable already explained by the other variables?*

$$VIF_j = \frac{1}{1 - R^2_j}$$

The rule is simple:

| VIF | Decision |
|---|---|
| < 5 | Keep |
| 5 – 10 | Tolerate |
| > 10 | Remove |

We compute VIF for every feature and drop the ones above 10.


In [ ]:
# Variance Inflation Factor — removes redundant variables (VIF > 10 means the variable
# is already explained by the others and adds no new information to the model)
VIF = np.diag(np.linalg.inv(np.corrcoef(final_features[:, :-1].T)))
for name, vif in zip(features.columns[:-1], VIF):
    print(f"{name:20} : VIF = {vif:.1f}")

In [ ]:
# Remove all features with VIF > 10 — only independent vocal biomarkers are kept
features = features.drop(columns=[
    'Jitter(%)', 'Jitter:RAP', 'Jitter:PPQ5',
    'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3',
    'Shimmer:APQ5', 'Shimmer:APQ11'
])

# Rebuild the matrix with clean features only
final_features           = np.array(features)
Transpose_final_features = np.transpose(final_features)

print("Features kept after VIF filter:")
for col in features.columns:
    print(f"  - {col}")

## 5. OLS Regression — Built From Scratch

Rather than using a library like `sklearn`, the model is derived directly from the **Normal Equation**:

$$\hat{\beta} = (X^TX)^{-1}X^Ty$$

This formula gives us the coefficients that minimize the sum of squared errors between the predicted and actual motor UPDRS scores. Building it manually makes every step transparent and shows a real understanding of what the model is doing under the hood.

Each coefficient $\hat{\beta}_j$ tells us: *if this vocal biomarker increases by 1 standard deviation, by how much does the predicted motor UPDRS score change?*


In [ ]:
# OLS Normal Equation — beta = (XᵀX)⁻¹ Xᵀy
# Each coefficient measures the impact of one biomarker on motor UPDRS severity
beta_hat = (np.linalg.inv(Transpose_final_features @ final_features)) @ Transpose_final_features @ final_target_variable

y_hat     = final_features @ beta_hat
residuals = final_target_variable - y_hat

# R² measures how much of the variance in motor UPDRS is explained by vocal biomarkers
ss_res = np.sum(residuals ** 2)
ss_tot = np.sum((final_target_variable - final_target_variable.mean()) ** 2)
r2     = 1 - ss_res / ss_tot
rmse   = np.sqrt(np.mean(residuals ** 2))

columns_names  = features.columns
feature_names  = list(columns_names)
interpretation = dict(zip(columns_names, beta_hat))

print("OLS Coefficients:")
for feature, coef in interpretation.items():
    print(f"  {feature:20} : {coef:+.4f}")
print(f"\nR²   = {r2:.4f}")
print(f"RMSE = {rmse:.4f}")

## 6. Results

Three charts summarize the model output:

- **OLS Coefficients** : red bars increase motor severity, green bars are associated with lower severity
- **Predicted vs Actual** : each dot is one recording; the closer to the diagonal, the better the prediction
- **Residual Distribution** : checks whether prediction errors are centered around zero, which is a core assumption of OLS


In [ ]:
sns.set_theme(style='whitegrid', font='serif')

# One large chart on top (coefficients) and two smaller charts below (diagnostics)
fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.3)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

# Red = increases severity / Green = associated with lower severity
coef_df = pd.DataFrame({
    'feature':     [k for k in interpretation if k != 'Intercept'],
    'coefficient': [v for k, v in interpretation.items() if k != 'Intercept']
}).sort_values('coefficient')

colors = ['#E63946' if v > 0 else '#2A9D8F' for v in coef_df['coefficient']]

sns.barplot(data=coef_df, x='coefficient', y='feature',
            palette=colors, orient='h', ax=ax1)
ax1.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax1.set_title("OLS Coefficients — Vocal Biomarkers vs Motor UPDRS", fontweight='bold')
ax1.set_xlabel("OLS Coefficient (standardized features)")
ax1.set_ylabel("")

# Points close to the red dashed line = accurate predictions
sns.scatterplot(x=final_target_variable, y=y_hat,
                alpha=0.25, s=12, color='#2E4057', ax=ax2)
lims = [final_target_variable.min(), final_target_variable.max()]
ax2.plot(lims, lims, color='#E63946', linestyle='--',
         linewidth=1.5, label='Perfect fit')
ax2.set_xlabel("Actual motor UPDRS")
ax2.set_ylabel("Predicted motor UPDRS")
ax2.set_title(f"Predicted vs Actual — R² = {r2:.3f} | RMSE = {rmse:.3f}",
              fontweight='bold')
ax2.legend(fontsize=9)

# Residuals should be centered around zero and bell-shaped for OLS to be valid
sns.histplot(residuals, bins=50, kde=True,
             color='#2E4057', stat='density', ax=ax3)
ax3.axvline(x=0, color='#E9C46A', linestyle='--',
            linewidth=1.5, label='Zero line')
ax3.set_xlabel("Residuals")
ax3.set_ylabel("Density")
ax3.set_title("Residual Distribution — OLS Assumption Check", fontweight='bold')
ax3.legend(fontsize=9)

fig.suptitle(
    "Parkinson's Disease Severity — OLS Regression Analysis\n"
    f"n = 5,875 observations  |  R² = {r2:.3f}  |  RMSE = {rmse:.3f}",
    fontsize=13, fontweight='bold', y=0.98
)

plt.savefig('parkinsons_ols_analysis.png', dpi=300,
            facecolor='white', bbox_inches='tight')
plt.show()

## 7. Conclusion

After correcting for multicollinearity using the Variance Inflation Factor (VIF > 10), our OLS model identifies **PPE** as the main vocal biomarker influencing Parkinson's severity.

**Clinically speaking:**  
PPE (Pitch Period Entropy) represents the chaos and irregularity in the patient's voice. It captures something that other biomarkers do not  the unpredictability of vocal patterns, which reflects the underlying loss of motor control.

**Based on our results**, PPE is the number 1 predictor of motor symptom aggravation in this dataset. The low R² (0.092) suggests that vocal biomarkers alone have limited predictive power, other clinical features would be needed to fully explain motor UPDRS severity.

---

### What This Tells Us

The voice carries real clinical information. Even with a simple linear model built entirely from scratch, we can detect a statistically meaningful signal between vocal patterns and motor symptoms.

The limitation is not the method — it is the data. A linear model can only go so far when the underlying relationship is complex and non-linear. The natural next step would be to test a **Random Forest** or a **Ridge Regression** on the same cleaned features, and compare the results.

---

### Limitations

| Issue | What it means |
|---|---|
| R² = 0.092 | Vocal features explain only 9% of motor severity variance |
| Non-normal residuals | The relationship is likely non-linear |
| 42 patients, 5,875 recordings | Observations are not independent — same patient appears many times |
